# VotingAgent Example with DSPy

This notebook demonstrates the `VotingAgent` from `aap_core.orchestration` using DSPy chains.

**Topic:** Which shape should we choose to package beverages like milk, soft drink or beer?

Multiple agents will independently generate answers, then the `VotingAgent` will select the best one using BLEU/ROUGE scoring.

In [ ]:
from typing import List

from a2a.types import AgentCapabilities, AgentCard, AgentSkill
from aap_core.agent import BaseAgent
from aap_core.orchestration import VotingAgent
from aap_core.types import AgentMessage, AgentResponse, BaseLLMChain
from aap_dspy.chain import BaseSignatureAdapter, ChatCausalMultiTurnsChain

import dspy

In [3]:
class BeverageShapeSignature(dspy.Signature):
    """Signature for recommending beverage packaging shapes."""
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


class SignatureAdapter(BaseSignatureAdapter[BeverageShapeSignature]):
    def msg2sig(self, message: AgentMessage) -> List[BeverageShapeSignature]:
        bev_shape = BeverageShapeSignature(question=message.query, answer="")
        return [bev_shape]

    def sig2msg(self, signatures: List[BeverageShapeSignature], name: str) -> List[AgentResponse]:
        responses = []
        for sig in signatures:
            responses.append((name, sig.answer))
        return responses

In [4]:
class Agent(BaseAgent):
    chain: BaseLLMChain

    def execute(self, message: AgentMessage, **kwargs) -> AgentMessage:
        self.state = "running"
        message = self.chain.invoke(message, **kwargs)
        message.execution_result = "success"
        message.origin = self.card.name
        self.state = "idle"
        return message

In [18]:
# Define system prompts for each agent with different perspectives
prompts = {
    "packaging_engineer": "You are a packaging engineer. Focus on material science, manufacturing efficiency, and structural integrity when recommending beverage packaging shapes.",
    "marketing_expert": "You are a marketing expert. Focus on consumer appeal, shelf presence, brand differentiation, and market trends when recommending beverage packaging shapes.",
    "sustainability_consultant": "You are a sustainability consultant. Focus on environmental impact, recyclability, carbon footprint, and circular economy principles when recommending beverage packaging shapes.",
}

# Create agent cards
agent_cards = {}
for name, desc in prompts.items():
    skill = AgentSkill(
        id=f'{name}-skill',
        name=f'{name} skill',
        description=desc,
        tags=[name]
    )
    agent_cards[name] = AgentCard(
        name=name,
        description=desc,
        skills=[skill],
        capabilities=AgentCapabilities(),
        default_input_modes=['text'],
        default_output_modes=['text'],
        url="localhost",
        version="0.1.0"
    )

# Voting agent card
voting_skill = AgentSkill(
    id='voting-skill',
    name='voting skill',
    description='Voting agent that selects the best answer among candidates',
    tags=['voting']
)
voting_card = AgentCard(
    name='voting_agent',
    description='Voting agent that selects the best answer among candidates',
    skills=[voting_skill],
    capabilities=AgentCapabilities(),
    default_input_modes=['text'],
    default_output_modes=['text'],
    url="localhost",
    version="0.1.0"
)

# Setup Ollama LM
model = dspy.LM(
    model="ollama_chat/mistral:7b-instruct-v0.3-q4_K_S",
    api_base="http://localhost:11434",
    api_key="",
    temperature=0.8,
    cache=False
)

# Create agents with different system prompt perspectives
agents = []
for name, system_prompt in prompts.items():
    adapter = SignatureAdapter.with_prefill({"guide": system_prompt})
    predictor = dspy.ChainOfThought(BeverageShapeSignature)
    chain = ChatCausalMultiTurnsChain(
        BeverageShapeSignature,
        predictor=predictor,
        adapter=adapter
    ).with_lm(model)
    # Set chain name to match agent card name so VotingAgent can match responses
    chain.name = name
    agent = Agent(chain=chain, card=agent_cards[name])
    agents.append(agent)
    print(f"Created agent: {name}")

Created agent: packaging_engineer
Created agent: marketing_expert
Created agent: sustainability_consultant


In [19]:
# Create the VotingAgent using majority_vote with BLEU scoring
voting_agent = VotingAgent(
    agents=agents,
    voting_method="majority_vote",
    scorer="bleu",
    card=voting_card
)
print(f"VotingAgent created with {len(agents)} agents, voting method: majority_vote, scorer: bleu")

VotingAgent created with 3 agents, voting method: majority_vote, scorer: bleu


In [20]:
# Define the query
query = "Which shape should we choose to package beverages like milk, soft drink or beer? Consider factors like shelf stability, consumer convenience, manufacturing cost, and environmental impact. Provide a detailed recommendation."
message = AgentMessage(query=query)
print(f"Query: {query}")
print("-" * 80)

Query: Which shape should we choose to package beverages like milk, soft drink or beer? Consider factors like shelf stability, consumer convenience, manufacturing cost, and environmental impact. Provide a detailed recommendation.
--------------------------------------------------------------------------------


In [22]:
# Debug: Check chain names vs agent card names
print("=== DEBUG: Name matching ===")
for i, agent in enumerate(agents):
    print(f"Agent {i}: card.name = '{agent.card.name}', chain.name = '{agent.chain.name}'")
    print(f"  Match: {agent.card.name == agent.chain.name}")

# Create the query
query = "Which shape should we choose to package beverages like milk, soft drink or beer? Consider factors like shelf stability, consumer convenience, manufacturing cost, and environmental impact. Provide a detailed recommendation."
message = AgentMessage(query=query)
print(f"\nQuery: {query}")
print("-" * 80)

=== DEBUG: Name matching ===
Agent 0: card.name = 'packaging_engineer', chain.name = 'packaging_engineer'
  Match: True
Agent 1: card.name = 'marketing_expert', chain.name = 'marketing_expert'
  Match: True
Agent 2: card.name = 'sustainability_consultant', chain.name = 'sustainability_consultant'
  Match: True

Query: Which shape should we choose to package beverages like milk, soft drink or beer? Consider factors like shelf stability, consumer convenience, manufacturing cost, and environmental impact. Provide a detailed recommendation.
--------------------------------------------------------------------------------


In [24]:
# Step 1: Execute all agents to generate candidate responses
print("=== Step 1: Executing all agents ===")
for agent in agents:
    test_msg = AgentMessage(query=query)
    result = agent.execute(test_msg)
    print(f"Agent '{agent.card.name}': response generated")
    print(f"  Response name: '{result.responses[0][0]}'")
    print(f"  Response preview: {result.responses[0][1][:200]}...")
    print()

# Step 2: Collect all responses into a single message for voting
print("=== Step 2: Collecting responses for voting ===")
voting_message = AgentMessage(query=query)
for agent in agents:
    test_msg = AgentMessage(query=query)
    result = agent.execute(test_msg)
    voting_message.responses.append(result.responses[-1])  # Append the last response from each agent

print(f"Collected {len(voting_message.responses)} candidate responses")
for resp_name, resp_content in voting_message.responses:
    print(f"  - {resp_name}")
print()

# Step 3: Execute VotingAgent to select the best response
print("=== Step 3: Executing VotingAgent ===")
result = voting_agent.execute(voting_message)
print(f"Execution result: {result.execution_result}")
print(f"Number of responses after voting: {len(result.responses)}")
print("-" * 80)

=== Step 1: Executing all agents ===
Agent 'packaging_engineer': response generated
  Response name: 'packaging_engineer'
  Response preview: Given the factors mentioned, I recommend using either a bottle or can for packaging milk, soft drinks, or beer, depending on specific branding needs and whether light protection is required (for examp...

Agent 'marketing_expert': response generated
  Response name: 'marketing_expert'
  Response preview: Considering factors such as shelf stability, consumer convenience, manufacturing cost, and environmental impact, it is recommended to use a combination of rectangular, cylindrical, and opaque packagin...

Agent 'sustainability_consultant': response generated
  Response name: 'sustainability_consultant'
  Response preview: Given the factors discussed above, I recommend choosing a cylindrical or rectangular shape to package beverages like milk, soft drinks, or beer. These shapes offer good shelf stability, consumer conve...

=== Step 2: Collecting 

In [25]:
# Display all candidate responses
print("=== CANDIDATE RESPONSES ===")
print()
for agent_name, response in result.responses:
    print(f"--- Agent: {agent_name} ---")
    print(response)
    print()
    print("-" * 80)
    print()

=== CANDIDATE RESPONSES ===

--- Agent: packaging_engineer ---
Based on factors such as shelf stability, consumer convenience, manufacturing cost, and environmental impact, a rectangular container with handle holes appears to be the most suitable shape for packaging beverages like milk, soft drinks, or beer. This design offers good stability, is easy to handle, can potentially reduce manufacturing costs, and is environmentally friendly due to its efficient space utilization during storage and transportation.

--------------------------------------------------------------------------------

--- Agent: marketing_expert ---
Considering factors such as shelf stability, consumer convenience, manufacturing cost, and environmental impact, the most suitable shapes for packaging beverages like milk are aseptic cartons (like gable-top or Tetra Pak), soft drinks in PET bottles, and beer either in glass bottles or aluminum cans. The choice between these packaging options depends on factors specifi

In [26]:
# Display the winning response (selected by VotingAgent)
print("=== WINNING RESPONSE (selected by VotingAgent) ===")
print()
winner_name, winner_response = result.responses[-1]
print(f"Winner: {winner_name}")
print(winner_response)

=== WINNING RESPONSE (selected by VotingAgent) ===

Winner: voting_agent
Based on factors such as shelf stability, consumer convenience, manufacturing cost, and environmental impact, a rectangular container with handle holes appears to be the most suitable shape for packaging beverages like milk, soft drinks, or beer. This design offers good stability, is easy to handle, can potentially reduce manufacturing costs, and is environmentally friendly due to its efficient space utilization during storage and transportation.


In [27]:
# Inspect dspy history for all agent calls
dspy.inspect_history(n=3)





[2026-05-14T15:30:56.305186]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Signature for recommending beverage packaging shapes.


User message:

[[ ## question ## ]]
Which shape should we choose to package beverages like milk, soft drink or beer? Consider factors like shelf stability, consumer convenience, manufacturing cost, and environmental impact. Provide a detailed recommendation.

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reasoning ## ]]
In choosing a shape for bever